# Claims Aging Pipeline
**Closed Summary · Open Aging Pareto · Open Tasks**

Single-entry pipeline: `data/raw → interim → processed → exports`  
All business logic lives in `src/`. This notebook is the orchestrator.

In [1]:
# ── STEP 1 / 8 — Imports ────────────────────────────────────────────────
import sys
from pathlib import Path

# Add project root to sys.path so `src` is importable from notebooks/
_root = Path.cwd()
for _p in [_root, *_root.parents]:
    if (_p / "src").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

import yaml
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')          # non-interactive backend for saved PNGs
import matplotlib.pyplot as plt

from src.paths      import get_paths
from src.loaders    import load_excel, normalize_columns
from src.transforms import (
    filter_by_warehouses_and_dates,
    closed_claims_summary,
    open_claims_aging_table,
    build_aging_summary,
    build_open_tasks,
)
from src.exporters  import plot_pareto_for_wh, export_claims_workbook

print("[STEP 1/8] Imports complete")

[STEP 1/8] Imports complete


In [2]:
# ── STEP 2 / 8 — Load Configuration ─────────────────────────────────────
P = get_paths()

with open(P['config'] / 'settings.yaml') as f:
    cfg = yaml.safe_load(f)

warehouses   = cfg['warehouses']
start_date   = cfg['start_date']
end_date     = cfg['end_date']
max_labels   = cfg['max_labels']
input_files  = [(item['path'], item['name']) for item in cfg['input_files']]
output_excel = cfg['output_excel']

print(f"[STEP 2/8] Config loaded — {len(warehouses)} warehouses, "
      f"date range {start_date} → {end_date}, "
      f"{len(input_files)} input file(s)")

[STEP 2/8] Config loaded — 8 warehouses, date range 2024-01-01 → None, 2 input file(s)


In [3]:
# ── STEP 3 / 8 — Resolve Paths ──────────────────────────────────────────
# P was initialized in Step 2 alongside config.
# Confirm all required directories exist.
for key in ('raw', 'exports', 'charts'):
    P[key].mkdir(parents=True, exist_ok=True)

print(f"[STEP 3/8] Paths resolved — root: {P['root']}")
print(f"           raw:     {P['raw']}")
print(f"           exports: {P['exports']}")
print(f"           charts:  {P['charts']}")

[STEP 3/8] Paths resolved — root: C:\Users\dbalan\Desktop\Python\Github\2. Claims
           raw:     C:\Users\dbalan\Desktop\Python\Github\2. Claims\data\raw
           exports: C:\Users\dbalan\Desktop\Python\Github\2. Claims\data\exports
           charts:  C:\Users\dbalan\Desktop\Python\Github\2. Claims\figs_claims


In [4]:
# ── STEP 4 / 8 — Load Data ──────────────────────────────────────────────
raw_frames = {}   # dataset_name → raw DataFrame

for rel_path, dname in input_files:
    full_path = P['raw'] / rel_path
    if not full_path.exists():
        print(f"  WARNING: File not found, skipping: {full_path}")
        continue
    raw_frames[dname] = load_excel(full_path)
    print(f"  Loaded '{dname}' — {len(raw_frames[dname]):,} rows from {full_path.name}")

print(f"[STEP 4/8] Load complete — {len(raw_frames)} dataset(s) loaded")

  Loaded 'Ford Claims' — 4,419 rows from Ford Claims Planner.xlsx


  Loaded 'Chrysler Claims' — 4,044 rows from Chrysler Claims Planner.xlsx
[STEP 4/8] Load complete — 2 dataset(s) loaded


In [5]:
# ── STEP 5 / 8 — Clean / Validate ───────────────────────────────────────
# normalize_columns() standardizes column names and casts types.
# filter_by_warehouses_and_dates() applies warehouse list + date range.

normalized  = {}   # dataset_name → (df_std, df_filtered)
combined_rows = []

for dname, raw_df in raw_frames.items():
    df_std, rename_map = normalize_columns(raw_df)

    # Merge standardized columns back onto original (preserves extra columns)
    df_comb = raw_df.copy()
    for col in ['Warehouse', 'Start Date', 'Completed Date',
                'Task Name', 'Labels', 'Due Date']:
        if col in df_std.columns:
            df_comb[col] = df_std[col]
    df_comb['Dataset'] = dname
    combined_rows.append(df_comb)

    df_filt = filter_by_warehouses_and_dates(df_std, warehouses, start_date, end_date)
    normalized[dname] = (df_std, df_filt)
    print(f"  '{dname}': {len(df_std):,} rows normalized → {len(df_filt):,} in scope")

print(f"[STEP 5/8] Clean/Validate complete — {len(normalized)} dataset(s) processed")

  'Ford Claims': 4,419 rows normalized → 1,372 in scope
  'Chrysler Claims': 4,044 rows normalized → 1,299 in scope
[STEP 5/8] Clean/Validate complete — 2 dataset(s) processed


In [6]:
%%time
# ── STEP 6 / 8 — Transform / Analyze ────────────────────────────────────

closed_summaries   = []
open_aging_tables  = []
open_task_rows     = []

for dname, (df_std, df_filt) in normalized.items():
    closed_summaries.append(closed_claims_summary(df_filt, dname))
    open_aging_tables.append(open_claims_aging_table(df_filt, dname, warehouses))
    open_task_rows.append(build_open_tasks(df_filt, dname))

# Combined source rows (all datasets, filtered)
combined_all = pd.concat(combined_rows, ignore_index=True) if combined_rows else pd.DataFrame()
if not combined_all.empty:
    mask = combined_all['Warehouse'].notna()
    if start_date:
        mask &= pd.to_datetime(combined_all['Start Date'], errors='coerce') >= pd.to_datetime(start_date)
    if end_date:
        mask &= pd.to_datetime(combined_all['Start Date'], errors='coerce') <= pd.to_datetime(end_date)
    wanted = {w.strip().lower() for w in warehouses}
    mask &= combined_all['Warehouse'].astype(str).str.lower().isin(wanted)
    combined_filtered = combined_all[mask].copy()
else:
    combined_filtered = combined_all

closed_all = pd.concat(closed_summaries,  ignore_index=True) if closed_summaries  else pd.DataFrame()
aging_all  = pd.concat(open_aging_tables, ignore_index=True) if open_aging_tables else pd.DataFrame()

aging_summary = build_aging_summary(aging_all) if not aging_all.empty else pd.DataFrame()

open_tasks_detail = (
    pd.concat(open_task_rows, ignore_index=True)
    .sort_values(['Action Required', 'Due Date', 'Warehouse'], na_position='last')
    .reset_index(drop=True)
) if open_task_rows else pd.DataFrame(
    columns=['Dataset', 'Warehouse', 'Task Name', 'Action Required', 'Due Date']
)

open_tasks_agg = (
    open_tasks_detail
    .groupby(['Dataset', 'Warehouse', 'Action Required'], dropna=False)
    .size().rename('Count').reset_index()
    .sort_values(['Warehouse', 'Count'], ascending=[True, False], ignore_index=True)
) if not open_tasks_detail.empty else pd.DataFrame(
    columns=['Dataset', 'Warehouse', 'Action Required', 'Count']
)

print(f"[STEP 6/8] Transform complete")
print(f"           combined_filtered: {len(combined_filtered):,} rows")
print(f"           closed_all:        {len(closed_all):,} rows")
print(f"           aging_all:         {len(aging_all):,} rows")
print(f"           open_tasks_detail: {len(open_tasks_detail):,} rows")

[STEP 6/8] Transform complete
           combined_filtered: 2,671 rows
           closed_all:        16 rows
           aging_all:         64 rows
           open_tasks_detail: 58 rows
CPU times: total: 31.2 ms
Wall time: 29 ms


In [7]:
# ── STEP 7 / 8 — Visualize ──────────────────────────────────────────────
# Pareto charts saved to charts/ (one PNG per dataset × warehouse).

chart_count = 0

if not aging_all.empty:
    desired_order = ['<30 days', '30-<60 days', '60-<90 days', '>= 90 days']
    for dname in aging_all['Dataset'].unique():
        sub = aging_all[aging_all['Dataset'] == dname]
        for wh in warehouses:
            wh_rows = sub[sub['Warehouse'] == wh]
            if wh_rows.empty:
                continue
            wh_rows = (
                wh_rows.set_index('Aging Bucket')
                .reindex(desired_order)
                .reset_index()
            )
            plot_pareto_for_wh(
                wh_rows, dname, wh,
                max_labels=max_labels,
                charts_path=P['charts'],
            )
            chart_count += 1
else:
    print('  No aging data — charts skipped.')

print(f"[STEP 7/8] Visualize complete — {chart_count} chart(s) saved to {P['charts']}")

[STEP 7/8] Visualize complete — 16 chart(s) saved to C:\Users\dbalan\Desktop\Python\Github\2. Claims\figs_claims


In [8]:
# ── STEP 8 / 8 — Export Outputs ─────────────────────────────────────────
# Single multi-sheet workbook written to data/exports/.

out_path = export_claims_workbook(
    combined_filtered=combined_filtered,
    closed_all=closed_all,
    aging_all=aging_all,
    aging_summary=aging_summary,
    open_tasks_detail=open_tasks_detail,
    open_tasks_agg=open_tasks_agg,
    exports_path=P['exports'],
    filename=output_excel,
)

print(f"[STEP 8/8] Export complete")
print(f"           Workbook: {out_path}")
print(f"           Sheets  : All_Claims_Combined, Closed_Claims_Summary, "
      f"Open_Claims_Aging, Open_Aging_Summary, "
      f"Open_Tasks_By_Warehouse, Open_Tasks_Aggregate")

[STEP 8/8] Export complete
           Workbook: C:\Users\dbalan\Desktop\Python\Github\2. Claims\data\exports\claims_analysis_output.xlsx
           Sheets  : All_Claims_Combined, Closed_Claims_Summary, Open_Claims_Aging, Open_Aging_Summary, Open_Tasks_By_Warehouse, Open_Tasks_Aggregate


In [9]:
# ── STEP 9 / 9 — Regenerate Dashboard HTML ──────────────────────────────
# Splices fresh data into dashboard.html between @@DATA_START@@ / @@DATA_END@@.
# Writes both the root dashboard.html and a dated archive copy in dashboards/.

import json as _json
import re as _re
import numpy as _np
from datetime import date as _date

def _j(v):
    if isinstance(v, _np.integer):  return int(v)
    if isinstance(v, _np.floating): return float(v)
    return v

def _summary_rows(df):
    rows = []
    for _, r in df.sort_values('Total Open Claims', ascending=False).iterrows():
        rows.append({
            'wh':    str(r['Warehouse']),
            'lt30':  _j(r['<30 days']),
            'd30':   _j(r['30-<60 days']),
            'd60':   _j(r['60-<90 days']),
            'ge90':  _j(r['>= 90 days']),
            'total': _j(r['Total Open Claims']),
            'pct90': round(float(r['% >= 90 days']) * 100, 1) if r['Total Open Claims'] > 0 else 0,
        })
    return rows

def _oldest_rows(dataset_name, n=10):
    df = combined_filtered[combined_filtered['Dataset'] == dataset_name].copy()
    open_df = df[df['Completed Date'].isna()].copy()
    today = pd.Timestamp('today').normalize()
    open_df['days_open'] = (today - pd.to_datetime(open_df['Start Date'], errors='coerce')).dt.days
    top = open_df.sort_values('days_open', ascending=False).head(n)
    rows = []
    for _, r in top.iterrows():
        parts = str(r['Task Name']).split(' - ')
        label = ' – '.join(parts[:3]) if len(parts) >= 3 else str(r['Task Name'])[:70]
        rows.append({
            'task':      label,
            'warehouse': str(r['Warehouse']),
            'days':      int(r['days_open']) if pd.notna(r['days_open']) else 0,
            'start':     str(r['Start Date'])[:10] if pd.notna(r['Start Date']) else '',
        })
    return rows

def _action_rows(dataset_name):
    sub = open_tasks_agg[open_tasks_agg['Dataset'] == dataset_name].sort_values('Count', ascending=False)
    return [{'wh': str(r['Warehouse']), 'action': str(r['Action Required']), 'count': _j(r['Count'])}
            for _, r in sub.iterrows()]

def _task_detail_rows(dataset_name):
    sub = open_tasks_detail[open_tasks_detail['Dataset'] == dataset_name]
    rows = []
    for _, r in sub.iterrows():
        rows.append({
            'wh':     str(r['Warehouse']),
            'action': str(r['Action Required']),
            'task':   str(r['Task Name']),
            'due':    str(r['Due Date'])[:10] if pd.notna(r['Due Date']) else '',
        })
    return rows

ford_sum          = _summary_rows(aging_summary[aging_summary['Dataset'] == 'Ford Claims'])
chrys_sum         = _summary_rows(aging_summary[aging_summary['Dataset'] == 'Chrysler Claims'])
ford_old          = _oldest_rows('Ford Claims')
chrys_old         = _oldest_rows('Chrysler Claims')
ford_acts         = _action_rows('Ford Claims')
chrys_acts        = _action_rows('Chrysler Claims')
ford_task_detail  = _task_detail_rows('Ford Claims')
chrys_task_detail = _task_detail_rows('Chrysler Claims')

ford_total        = sum(r['total'] for r in ford_sum)
chrys_total       = sum(r['total'] for r in chrys_sum)
ford_ge90         = sum(r['ge90']  for r in ford_sum)
ford_6089         = sum(r['d60']   for r in ford_sum)
chrys_ge90        = sum(r['ge90']  for r in chrys_sum)
ford_biggest      = max(ford_sum,  key=lambda r: r['total']) if ford_sum  else {'wh': '', 'total': 0}
chrys_biggest     = max(chrys_sum, key=lambda r: r['total']) if chrys_sum else {'wh': '', 'total': 0}
ford_oldest_days  = max((r['days'] for r in ford_old),  default=0)
chrys_oldest_days = max((r['days'] for r in chrys_old), default=0)
ford_oldest_wh    = next((r['warehouse'] for r in ford_old  if r['days'] == ford_oldest_days),  '')
chrys_oldest_wh   = next((r['warehouse'] for r in chrys_old if r['days'] == chrys_oldest_days), '')

as_of = _date.today().strftime('%B %d, %Y')

data_js = '\n'.join([
    f'const AS_OF             = {_json.dumps(as_of)};',
    f'const fordSummary       = {_json.dumps(ford_sum,          indent=2)};',
    f'const chrysSummary      = {_json.dumps(chrys_sum,         indent=2)};',
    f'const fordOldest        = {_json.dumps(ford_old,          indent=2)};',
    f'const chrysOldest       = {_json.dumps(chrys_old,         indent=2)};',
    f'const fordActions       = {_json.dumps(ford_acts,         indent=2)};',
    f'const chrysActions      = {_json.dumps(chrys_acts,        indent=2)};',
    f'const fordTaskDetails   = {_json.dumps(ford_task_detail,  indent=2)};',
    f'const chrysTaskDetails  = {_json.dumps(chrys_task_detail, indent=2)};',
    f'const fordTotal         = {ford_total};',
    f'const chrysTotal        = {chrys_total};',
    f'const fordGe90          = {ford_ge90};',
    f'const ford6089          = {ford_6089};',
    f'const chrysGe90         = {chrys_ge90};',
    f'const fordBiggestWh     = {_json.dumps(ford_biggest["wh"])};',
    f'const fordBiggestN      = {ford_biggest["total"]};',
    f'const chrysBiggestWh    = {_json.dumps(chrys_biggest["wh"])};',
    f'const chrysBiggestN     = {chrys_biggest["total"]};',
    f'const fordOldestDays    = {ford_oldest_days};',
    f'const fordOldestWh      = {_json.dumps(ford_oldest_wh)};',
    f'const chrysOldestDays   = {chrys_oldest_days};',
    f'const chrysOldestWh     = {_json.dumps(chrys_oldest_wh)};',
])

replacement = '// @@DATA_START@@\n' + data_js + '\n// @@DATA_END@@'
template_path = P['root'] / 'dashboard.html'
html = template_path.read_text(encoding='utf-8')
html = _re.sub(r'// @@DATA_START@@.*?// @@DATA_END@@', lambda _: replacement, html, flags=_re.DOTALL)

# Overwrite root dashboard.html
template_path.write_text(html, encoding='utf-8')

# Save dated archive copy
P['dashboards'].mkdir(parents=True, exist_ok=True)
dated_path = P['dashboards'] / f"{_date.today().strftime('%Y%m%d')}_claims_aging_dashboard.html"
dated_path.write_text(html, encoding='utf-8')

print(f"[STEP 9/9] Dashboard updated")
print(f"           Root:    {template_path}")
print(f"           Archive: {dated_path}")

[STEP 9/9] Dashboard updated
           Root:    C:\Users\dbalan\Desktop\Python\Github\2. Claims\dashboard.html
           Archive: C:\Users\dbalan\Desktop\Python\Github\2. Claims\dashboards\20260731_claims_aging_dashboard.html


In [10]:
# ── Preview — Aging Summary (leadership view) ────────────────────────────
if not aging_summary.empty:
    display(
        aging_summary
        .sort_values(['Dataset', 'Total Open Claims'], ascending=[True, False])
        .reset_index(drop=True)
        .style.format('{:.1%}', subset=['% >= 90 days'])
    )

,Dataset,Warehouse,<30 days,30-<60 days,60-<90 days,>= 90 days,Total Open Claims,% >= 90 days
0,Chrysler Claims,Charlotte,10,5,1,25,41,61.0%
1,Chrysler Claims,Ontario,7,5,3,23,38,60.5%
2,Chrysler Claims,Atlanta,8,10,2,13,33,39.4%
3,Chrysler Claims,Orlando,8,1,2,19,30,63.3%
4,Chrysler Claims,Phoenix,3,1,2,23,29,79.3%
5,Chrysler Claims,Flowood,1,2,4,19,26,73.1%
6,Chrysler Claims,El Paso,1,1,0,9,11,81.8%
7,Chrysler Claims,OKC,1,3,2,1,7,14.3%
8,Ford Claims,Atlanta,9,6,7,5,27,18.5%
9,Ford Claims,Orlando,8,3,1,7,19,36.8%
